# AeroAnalytics: Sales & Customer Exploratory Data Analysis (EDA)

This Jupyter Notebook demonstrates the backend data analysis workflow. We connect to our SQLite database `dashboard.db`, run SQL queries to compute business performance KPIs, load datasets into **Pandas DataFrames**, and construct visual plots with **Matplotlib**.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Database Connection & Table Inspect
Establish connection to the SQLite database generated by the project backend server.

In [ ]:
conn = sqlite3.connect('../data/dashboard.db')
cursor = conn.cursor()

# List tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in database:", [t[0] for t in tables])

## 2. Customer Segments Exploration
We pull customers grouped by segment, count records, and extract average transaction volumes.

In [ ]:
segment_query = """
SELECT 
    c.segment,
    COUNT(DISTINCT c.customer_id) as total_customers,
    ROUND(SUM(s.amount), 2) as gross_sales,
    ROUND(AVG(s.amount), 2) as avg_transaction_value
FROM customers c
LEFT JOIN sales s ON c.customer_id = s.customer_id
GROUP BY c.segment
ORDER BY gross_sales DESC;
"""

df_segments = pd.read_sql_query(segment_query, conn)
df_segments

## 3. Product Performance Analysis
Let's review which product categories generate the highest net profit margins.

In [ ]:
product_query = """
SELECT 
    category,
    SUM(quantity) as items_sold,
    ROUND(SUM(amount), 2) as total_sales,
    ROUND(SUM(profit), 2) as total_profit,
    ROUND((SUM(profit) / SUM(amount)) * 100, 2) as profit_margin_percent
FROM sales
GROUP BY category
ORDER BY total_sales DESC;
"""

df_products = pd.read_sql_query(product_query, conn)
df_products

In [ ]:
# Plotting Product Categories Sales vs Profit Margin
fig, ax1 = plt.subplots()

color = '#6366f1'
ax1.set_xlabel('Product Category')
ax1.set_ylabel('Total Revenue ($)', color=color)
ax1.bar(df_products['category'], df_products['total_sales'], color=color, alpha=0.85, width=0.5, label='Revenue')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_title('Product Category Sales Performance & Profit Margins')

ax2 = ax1.twinx()
color = '#10b981'
ax2.set_ylabel('Profit Margin (%)', color=color)
ax2.plot(df_products['category'], df_products['profit_margin_percent'], color=color, marker='o', linewidth=3, label='Margin %')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0, 100)

fig.tight_layout()
plt.show()

## 4. Monthly Sales & Profit Growth Trends
We aggregate sales and profits month-over-month to view seasonal growth trends.

In [ ]:
trend_query = """
SELECT 
    strftime('%Y-%m', transaction_date) as month,
    SUM(amount) as sales,
    SUM(profit) as profit
FROM sales
GROUP BY month
ORDER BY month;
"""

df_trend = pd.read_sql_query(trend_query, conn)
df_trend.set_index('month', inplace=True)
df_trend

In [ ]:
# Plot Monthly Revenue vs Profit trends
df_trend['sales'].plot(kind='line', marker='s', color='#3b82f6', linewidth=2, label='Monthly Revenue')
df_trend['profit'].plot(kind='line', marker='o', color='#10b981', linewidth=2, label='Monthly Net Profit')
plt.title('Monthly Sales Revenue & Profit Performance')
plt.xlabel('Fiscal Month')
plt.ylabel('Amount in USD ($)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Close Connection
Free up DB connection handles.

In [ ]:
conn.close()
print("Database connection closed successfully.")